# Malaika — Fine-Tune Gemma 4 on Audio Features as Text (Path 2)

**Key insight**: LoRA modifies the language model, not the vision encoder.
Instead of spectrograms (images → frozen SigLIP), we extract acoustic features
with librosa and pass them as text → LoRA can actually learn.

```
OLD: audio → spectrogram IMAGE → frozen SigLIP → LoRA → fails
NEW: audio → librosa NUMBERS → text prompt → LoRA → learns
```

**GPU**: T4 | **Secrets**: `HF_TOKEN`, `KAGGLE_USERNAME`, `KAGGLE_KEY`

In [ ]:
# Cell 1: Install
!pip install -q git+https://github.com/huggingface/transformers.git peft trl datasets bitsandbytes accelerate librosa soundfile kaggle

In [ ]:
# Cell 2: Auth + imports
from huggingface_hub import login
import os, subprocess, json, random, re, time, gc
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import torch
import librosa

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    login(token=secrets.get_secret("HF_TOKEN"))
    KU, KK = secrets.get_secret("KAGGLE_USERNAME"), secrets.get_secret("KAGGLE_KEY")
    print("Auth: Kaggle")
except ModuleNotFoundError:
    try:
        from google.colab import userdata
        login(token=userdata.get("HF_TOKEN"))
        KU, KK = userdata.get("KAGGLE_USERNAME"), userdata.get("KAGGLE_KEY")
        print("Auth: Colab")
    except Exception:
        login()
        KU, KK = os.environ.get("KAGGLE_USERNAME", ""), os.environ.get("KAGGLE_KEY", "")

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
    json.dump({"username": KU, "key": KK}, f)
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_mem/1024**3:.1f} GB")

In [ ]:
# Cell 3: Download ICBHI
ICBHI_DIR = Path("/tmp/icbhi_data")
ICBHI_INNER = ICBHI_DIR / "Respiratory_Sound_Database" / "Respiratory_Sound_Database"
KAGGLE_NATIVE = Path("/kaggle/input/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database")

if KAGGLE_NATIVE.exists():
    audio_dir = KAGGLE_NATIVE / "audio_and_txt_files"
elif ICBHI_INNER.exists():
    audio_dir = ICBHI_INNER / "audio_and_txt_files"
else:
    print("Downloading ICBHI...")
    ICBHI_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(["kaggle", "datasets", "download", "-d", "vbookshelf/respiratory-sound-database",
        "-p", str(ICBHI_DIR), "--unzip"], capture_output=True, text=True, check=True)
    audio_dir = ICBHI_INNER / "audio_and_txt_files"

audio_files = list(audio_dir.glob("*.wav"))
print(f"ICBHI: {len(audio_files)} audio files")

In [ ]:
# Cell 4: Parse annotations
def parse_icbhi_annotations(adir):
    records = []
    for txt in sorted(adir.glob("*.txt")):
        wav = txt.with_suffix(".wav")
        if not wav.exists(): continue
        pid = txt.stem.split("_")[0]
        hc, hw = False, False
        with open(txt) as f:
            for line in f:
                ff = line.strip().split()
                if len(ff) >= 4:
                    if int(ff[2]) == 1: hc = True
                    if int(ff[3]) == 1: hw = True
        label = "both" if hc and hw else "crackle" if hc else "wheeze" if hw else "normal"
        records.append({"audio_path": str(wav), "patient_id": pid,
            "label": label, "has_crackle": hc, "has_wheeze": hw})
    return records

records = parse_icbhi_annotations(audio_dir)
print(f"Parsed {len(records)} recordings:")
for label, count in sorted(Counter(r["label"] for r in records).items()):
    print(f"  {label}: {count}")

In [ ]:
# Cell 5: Extract acoustic features from every audio file
def extract_audio_features(audio_path, sr=22050):
    """Extract clinically-meaningful acoustic features using librosa."""
    y, sr = librosa.load(str(audio_path), sr=sr, mono=True)
    if len(y) == 0:
        return None
    duration = len(y) / sr

    # MFCCs — spectral shape (standard in audio ML)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfcc_mean = mfccs.mean(axis=1)
    mfcc_std = mfccs.std(axis=1)

    # Spectral features
    cent = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)[0]
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    rms = librosa.feature.rms(y=y)[0]
    contrast = librosa.feature.spectral_contrast(y=y, sr=sr, n_bands=6)
    contrast_mean = contrast.mean(axis=1)

    return {
        "duration": round(duration, 1),
        "mfcc_mean": [round(float(x), 1) for x in mfcc_mean],
        "mfcc_std": [round(float(x), 1) for x in mfcc_std],
        "spectral_centroid_mean": round(float(cent.mean()), 1),
        "spectral_centroid_std": round(float(cent.std()), 1),
        "spectral_bandwidth_mean": round(float(bw.mean()), 1),
        "spectral_bandwidth_std": round(float(bw.std()), 1),
        "spectral_rolloff_mean": round(float(rolloff.mean()), 1),
        "zero_crossing_rate_mean": round(float(zcr.mean()), 4),
        "zero_crossing_rate_std": round(float(zcr.std()), 4),
        "rms_energy_mean": round(float(rms.mean()), 4),
        "rms_energy_std": round(float(rms.std()), 4),
        "spectral_contrast": [round(float(x), 1) for x in contrast_mean],
    }

print("Extracting audio features...")
feat_records = []
for i, r in enumerate(records):
    try:
        feats = extract_audio_features(r["audio_path"])
        if feats:
            r["features"] = feats
            feat_records.append(r)
    except Exception as e:
        pass
    if (i+1) % 200 == 0: print(f"  {i+1}/{len(records)}")
records = feat_records
print(f"Extracted features from {len(records)} recordings")

# Show one example
print(f"\nExample ({records[0]['label']}):")
for k, v in records[0]["features"].items():
    print(f"  {k}: {v}")

In [ ]:
# Cell 6: Build text prompts + patient split
def features_to_text(feats):
    """Convert extracted features dict into a text prompt."""
    return (
        f"Audio features from a child's breathing recording ({feats['duration']}s):\n"
        f"- MFCC means: {feats['mfcc_mean']}\n"
        f"- MFCC stds: {feats['mfcc_std']}\n"
        f"- Spectral centroid: mean={feats['spectral_centroid_mean']} std={feats['spectral_centroid_std']}\n"
        f"- Spectral bandwidth: mean={feats['spectral_bandwidth_mean']} std={feats['spectral_bandwidth_std']}\n"
        f"- Spectral rolloff: mean={feats['spectral_rolloff_mean']}\n"
        f"- Zero crossing rate: mean={feats['zero_crossing_rate_mean']} std={feats['zero_crossing_rate_std']}\n"
        f"- RMS energy: mean={feats['rms_energy_mean']} std={feats['rms_energy_std']}\n"
        f"- Spectral contrast: {feats['spectral_contrast']}"
    )

INSTRUCTION = ("Analyze these acoustic features extracted from a child's breathing audio "
    "and classify the breath sounds.\n\n"
    "Feature guide:\n"
    "- Wheeze: higher spectral centroid, narrow bandwidth, low zero crossing rate (continuous tone)\n"
    "- Crackle: high zero crossing rate (discontinuous pops), variable centroid, wide bandwidth\n"
    "- Both: combination of wheeze and crackle features\n"
    "- Normal: low energy, moderate centroid, low contrast\n\n"
    "{features_text}\n\n"
    "Classify as exactly one word: normal, wheeze, crackle, or both")

pairs = []
for r in records:
    feat_text = features_to_text(r["features"])
    pairs.append({
        "instruction": INSTRUCTION.format(features_text=feat_text),
        "response": r["label"],
        "label": r["label"],
        "patient_id": r["patient_id"],
    })

# Patient-level 80/20 split
pids = list(set(r["patient_id"] for r in records))
random.seed(42); random.shuffle(pids)
train_pats = set(pids[:int(len(pids)*0.8)])
train_pairs = [p for p in pairs if p["patient_id"] in train_pats]
test_pairs = [p for p in pairs if p["patient_id"] not in train_pats]

print(f"Train: {len(train_pairs)} | Test: {len(test_pairs)}")
for name, sp in [("Train", train_pairs), ("Test", test_pairs)]:
    print(f"  {name}: {dict(Counter(p['label'] for p in sp))}")

print(f"\nSample prompt (truncated):")
print(train_pairs[0]["instruction"][:300])
print(f"...\nResponse: '{train_pairs[0]['response']}'")

In [ ]:
# Cell 7: Load model (TEXT ONLY — no vision needed!)
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

if 'model' in dir():
    del model; gc.collect(); torch.cuda.empty_cache()

MODEL_NAME = "google/gemma-4-E4B-it"
print(f"Loading {MODEL_NAME} (text-only mode)...")
t0 = time.monotonic()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto",
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4"),
    torch_dtype=torch.float16,
)
print(f"Loaded in {time.monotonic()-t0:.0f}s | VRAM: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

In [ ]:
# Cell 8: Add LoRA adapter
from peft import LoraConfig, get_peft_model, TaskType

model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()

# Unwrap Gemma4ClippableLinear if present
unwrapped = 0
for name, module in list(model.named_modules()):
    if module.__class__.__name__ == "Gemma4ClippableLinear":
        parts = name.split(".")
        parent = model
        for p in parts[:-1]: parent = getattr(parent, p)
        setattr(parent, parts[-1], module.linear)
        unwrapped += 1
if unwrapped: print(f"Unwrapped {unwrapped} layers")

model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.1, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type=TaskType.CAUSAL_LM,
))
model.print_trainable_parameters()

In [ ]:
# Cell 9: Build text-only dataset (no images, no vision encoder!)
from torch.utils.data import Dataset as TorchDataset

SYSTEM_MSG = ("You are a pediatric breath sound classifier. You receive acoustic features "
    "extracted from a child's breathing recording and classify them. "
    "Respond with exactly one word: normal, wheeze, crackle, or both.")

class AudioFeaturesDataset(TorchDataset):
    def __init__(self, pairs, tokenizer, max_length=512):
        self.pairs = pairs
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        pair = self.pairs[idx]
        # Build chat messages
        messages = [
            {"role": "user", "content": f"{SYSTEM_MSG}\n\n{pair['instruction']}"},
            {"role": "assistant", "content": pair["response"]},
        ]
        text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True,
                               max_length=self.max_length, padding="max_length")
        
        input_ids = inputs["input_ids"].squeeze(0)
        attention_mask = inputs["attention_mask"].squeeze(0)
        labels = input_ids.clone()
        
        # Mask everything except response
        resp_tokens = self.tokenizer.encode(pair["response"], add_special_tokens=False)
        resp_len = len(resp_tokens)
        if resp_len > 0 and len(labels) > resp_len:
            labels[:-resp_len] = -100
        # Also mask padding
        labels[attention_mask == 0] = -100
        
        return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

train_dataset = AudioFeaturesDataset(train_pairs, tokenizer)
sample = train_dataset[0]
print(f"Dataset: {len(train_dataset)} examples")
print(f"Sequence length: {sample['input_ids'].shape[0]}")
print(f"Response tokens: {(sample['labels'] != -100).sum().item()}")
print(f"\nNO pixel_values, NO image_position_ids — pure text!")

In [ ]:
# Cell 10: Train
from transformers import TrainingArguments, Trainer

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="./breath_sound_text_lora",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        max_steps=200,
        learning_rate=2e-5,
        warmup_steps=20,
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        fp16=True,
        logging_steps=10,
        save_steps=50,
        save_total_limit=3,
        seed=42,
        report_to="none",
    ),
    train_dataset=train_dataset,
)

print("Path 2: Audio features as text — NO vision encoder in the loop")
print(f"Train: {len(train_dataset)} | batch=2 | lr=2e-5 cosine | 200 steps")
t0 = time.monotonic()
result = trainer.train()
train_time = time.monotonic() - t0
print(f"\nDone in {train_time/60:.1f} min | Loss: {result.training_loss:.4f}")

In [ ]:
# Cell 11: Save adapter
ADAPTER_NAME = "malaika-breath-sounds-text-lora"
model.save_pretrained(ADAPTER_NAME)
tokenizer.save_pretrained(ADAPTER_NAME)
print(f"Saved to {ADAPTER_NAME}/")
for f in sorted(Path(ADAPTER_NAME).glob("*")):
    if f.is_file(): print(f"  {f.name}: {f.stat().st_size/1024/1024:.1f} MB")

In [ ]:
# Cell 12: Evaluate on test set
model.gradient_checkpointing_disable()
model.eval()

print("=" * 60)
print("EVALUATION — Audio Features as Text (Path 2)")
print("=" * 60)

correct, total_test = 0, 0
per_label = {"normal": [0,0], "wheeze": [0,0], "crackle": [0,0], "both": [0,0]}

for pair in test_pairs:
    messages = [{"role": "user", "content": f"{SYSTEM_MSG}\n\n{pair['instruction']}"}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=10, do_sample=False)
    pred_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip().lower()
    
    label = pair["label"]
    per_label[label][1] += 1
    pred_label = "unknown"
    for candidate in ["both", "crackle", "wheeze", "normal"]:
        if candidate in pred_text:
            pred_label = candidate
            break
    if pred_label == label:
        correct += 1; per_label[label][0] += 1; status = "PASS"
    else:
        status = f"FAIL (pred={pred_label})"
    total_test += 1
    if total_test <= 40:  # print first 40
        print(f"  {status:35s} [{label:>7s}] -> '{pred_text[:20]}'")

print(f"\n{'='*60}")
print(f"Overall: {correct}/{total_test} ({100*correct/total_test:.0f}%)")
print(f"Previous best: v5 spectrogram 40% | v2 spectrogram 50% (crackle only)")
print(f"\nPer-label:")
for label, (c, t) in per_label.items():
    if t > 0: print(f"  {label:>8s}: {c}/{t} ({100*c/t:.0f}%)")

In [ ]:
# Cell 13: Summary
print("=" * 60)
print("RESULTS — Audio Features as Text (Path 2)")
print("=" * 60)
print(f"""
## Approach: Acoustic Feature Extraction + Text Fine-Tuning

### Why This Works
LoRA adapts the language model's attention layers. Spectrograms go through
a frozen vision encoder that can't learn — so LoRA has nothing to work with.
By extracting audio features as numbers and passing them as text, the entire
pipeline goes through the text pathway where LoRA IS effective.

Inspired by VoRA (arxiv:2503.20680) — "skip the vision encoder, embed
visual/audio understanding directly in the language model."

### Features Extracted (per recording)
- 13 MFCCs (mean + std) — spectral shape
- Spectral centroid — sound brightness
- Spectral bandwidth — frequency range
- Spectral rolloff — energy distribution
- Zero crossing rate — discontinuity (crackle indicator)
- RMS energy — loudness
- Spectral contrast — peak/valley ratio

### Results
| Approach | Accuracy | Notes |
|----------|----------|-------|
| Baseline (no FT) | 25% | Predicts all normal |
| Spectrogram v2 | 50% | Crackle only (class bias) |
| Spectrogram v5 | 40% | 85% crackle, 2% normal |
| **Text features** | **{100*correct/total_test:.0f}%** | **This run** |

### Config
- Dataset: ICBHI 2017, {len(records)} recordings
- Train/Test: {len(train_pairs)}/{len(test_pairs)} (patient-level split)
- Model: Gemma 4 E4B-it (4-bit, text-only)
- LoRA: r=16, alpha=32, targets q/k/v/o_proj
- LR: 2e-5 cosine, 200 steps
- Training time: {train_time/60:.1f} min
""")
print("Per-label:")
for label, (c, t) in per_label.items():
    if t > 0: print(f"  {label}: {c}/{t} ({100*c/t:.0f}%)")